# 02 — Análisis de Calidad de Datos

**ILERNA Smart-Industry — Programación IA PAC DES**

Este notebook analiza la **calidad de los datos** que fluyen por el pipeline:

| Sección | Qué analiza |
|---------|-------------|
| **1. DLQ (Dead Letter Queue)** | Mensajes rechazados en `sensors_invalid` |
| **2. Hash Chain** | Integridad SHA256 de los mensajes verificados |
| **3. Completitud y nulos** | Campos ausentes o nulos en `sensors_clean` (MinIO) |
| **4. Distribución de unidades** | Proporción de mensajes en C / F / K |
| **5. Anomalías de validación** | Comparativa transformaciones.py vs datos reales |
| **6. Resumen ejecutivo** | Métricas globales de calidad |

---
**Prerequisitos:** pipeline corriendo, datos en InfluxDB y MinIO.
Ejecuta `source .devcontainer/init_pipeline.sh` si es la primera vez.

In [ ]:
import os, sys, json, warnings
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from confluent_kafka import Consumer, TopicPartition
from influxdb_client import InfluxDBClient
from influxdb_client.client.warnings import MissingPivotFunction

warnings.simplefilter('ignore', MissingPivotFunction)

# Añadir raíz del proyecto al path — funciona tanto desde notebooks/ como desde la raíz
_cwd = os.getcwd()
PROJECT_ROOT = _cwd if os.path.isdir(os.path.join(_cwd, 'src')) else os.path.abspath(os.path.join(_cwd, '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.processing.transformations import to_celsius, normalize_record
from src.processing.validators import validate_raw_message, VALID_UNITS
from src.processing.hash_chain import verify_chain, chain_integrity_report

---
## 1. Dead Letter Queue (DLQ) — `sensors_invalid`

El job `flink_hash_verifier_job` publica en `sensors_invalid` los mensajes que:
- Tienen el **hash roto** (alguien modificó el mensaje en tránsito, o fallo del sensor)
- Son **JSON inválido** (el bridge no los validó correctamente)

Analizamos cuántos hay, por qué motivo y qué máquinas los generan.

In [ ]:
def read_dlq(topic='sensors_invalid', max_msgs=500, timeout_sec=5):
    """Lee todos los mensajes del DLQ desde el offset 0."""
    msgs = []
    import time
    c = Consumer({
        'bootstrap.servers':  KAFKA_BROKER,
        'group.id':           'nb02-dlq-reader',
        'auto.offset.reset':  'earliest',
        'enable.auto.commit': False,
        'socket.timeout.ms':  3000,
        'session.timeout.ms': 6000,
    })
    c.assign([TopicPartition(topic, 0, 0)])
    deadline = time.time() + timeout_sec
    while len(msgs) < max_msgs and time.time() < deadline:
        msg = c.poll(0.5)
        if msg is None:
            break
        if msg.error():
            break
        try:
            msgs.append(json.loads(msg.value().decode('utf-8')))
        except Exception:
            pass
    c.close()
    return msgs

dlq_msgs = read_dlq(max_msgs=500)
print(f'Mensajes en DLQ (sensors_invalid): {len(dlq_msgs)}')
if dlq_msgs:
    print('Ejemplo:')
    print(json.dumps(dlq_msgs[0], indent=2))

In [ ]:
if not dlq_msgs:
    print('DLQ vacío — no hay mensajes rechazados. El pipeline procesa todos los mensajes correctamente.')
else:
    dlq_df = pd.DataFrame([
        {
            'device_id': m.get('device_id', m.get('raw', '?')[:20]),
            'reason':    m.get('reason', 'desconocido'),
            'ts':        m.get('ts', ''),
            'hash':      (m.get('hash', '') or '')[:16] + '…',
        }
        for m in dlq_msgs
    ])

    print(f'Total rechazados: {len(dlq_df)}')
    print('\nMotivos de rechazo:')
    print(dlq_df['reason'].value_counts().to_string())
    print('\nMáquinas con mensajes rechazados:')
    print(dlq_df['device_id'].value_counts().to_string())

    # Gráficos
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Motivos de rechazo', 'Rechazados por máquina'],
    )

    reason_counts = dlq_df['reason'].value_counts()
    fig.add_trace(
        go.Bar(x=reason_counts.values, y=reason_counts.index, orientation='h',
               marker_color='crimson', name='Motivo'),
        row=1, col=1
    )

    dev_counts = dlq_df['device_id'].value_counts()
    fig.add_trace(
        go.Bar(x=dev_counts.index, y=dev_counts.values,
               marker_color='darkorange', name='Máquina'),
        row=1, col=2
    )

    fig.update_layout(
        title='Análisis del Dead Letter Queue (sensors_invalid)',
        showlegend=False, height=400, template='plotly_dark',
    )
    fig.show()

---
## 2. Hash Chain — Verificación de Integridad

Leemos mensajes de `sensors_raw` directamente (no del DLQ) y verificamos
la cadena hash usando `src/02_processing/hash_chain.py`.

Esto permite hacer una **auditoría independiente** del job Flink.

In [ ]:
def read_topic(topic, max_msgs=200, timeout_sec=5):
    """Lee mensajes de un topic Kafka desde el offset 0."""
    import time
    msgs = []
    c = Consumer({
        'bootstrap.servers':  KAFKA_BROKER,
        'group.id':           f'nb02-reader-{topic}',
        'auto.offset.reset':  'earliest',
        'enable.auto.commit': False,
        'socket.timeout.ms':  3000,
    })
    c.assign([TopicPartition(topic, 0, 0)])
    deadline = time.time() + timeout_sec
    while len(msgs) < max_msgs and time.time() < deadline:
        msg = c.poll(0.5)
        if msg is None:
            break
        if msg.error():
            break
        try:
            msgs.append(json.loads(msg.value().decode('utf-8')))
        except Exception:
            pass
    c.close()
    return msgs

raw_msgs = read_topic('sensors_raw', max_msgs=300)
print(f'Mensajes leídos de sensors_raw: {len(raw_msgs)}')

In [ ]:
if not raw_msgs:
    print('No hay mensajes en sensors_raw. ¿Está el simulador corriendo?')
else:
    # Agrupar por device_id y verificar cadena por separado
    raw_df = pd.DataFrame(raw_msgs)
    devices = raw_df['device_id'].unique() if 'device_id' in raw_df.columns else []

    all_results = []
    for dev in devices:
        dev_msgs = raw_df[raw_df['device_id'] == dev].sort_values('ts').to_dict('records')
        results  = verify_chain(dev_msgs)
        report   = chain_integrity_report(results)
        all_results.append({
            'device_id':     dev,
            'total':         report['total'],
            'valid':         report['valid'],
            'invalid':       report['invalid'],
            'integrity_pct': report['integrity_pct'],
            'broken_at':     str(report['broken_at'][:5]) if report['broken_at'] else '—',
        })

    integrity_df = pd.DataFrame(all_results)
    print('Resumen de integridad por dispositivo:')
    print(integrity_df.to_string(index=False))

    # Gráfico
    fig_int = px.bar(
        integrity_df,
        x='device_id',
        y=['valid', 'invalid'],
        title='Integridad de la cadena hash por dispositivo',
        labels={'value': 'Mensajes', 'variable': 'Estado'},
        color_discrete_map={'valid': '#4caf50', 'invalid': '#f44336'},
        barmode='stack',
        template='plotly_dark',
    )
    fig_int.update_layout(height=380)
    fig_int.show()

    overall_valid   = integrity_df['valid'].sum()
    overall_total   = integrity_df['total'].sum()
    overall_pct     = overall_valid / overall_total * 100 if overall_total > 0 else 0
    print(f'\n→ Integridad global: {overall_valid}/{overall_total} ({overall_pct:.1f}%)')

---
## 3. Completitud y Nulos — Cold Path (MinIO)

Analizamos los ficheros NDJSON escritos por `kafka_to_minio.py` en MinIO:
- ¿Cuántos registros tienen campos nulos?
- ¿Qué proporción de registros son válidos?
- ¿Hay archivos con esquemas mixtos (migración)?

In [ ]:
conn = duckdb.connect()
conn.execute('INSTALL httpfs; LOAD httpfs;')
conn.execute(f"""
    SET s3_endpoint          = '{MINIO_ENDPOINT}';
    SET s3_access_key_id     = '{MINIO_ACCESS}';
    SET s3_secret_access_key = '{MINIO_SECRET}';
    SET s3_use_ssl           = false;
    SET s3_url_style         = 'path';
""")

try:
    minio_df = conn.execute(f"""
        SELECT
            device_id,
            CAST(temperature_c AS DOUBLE) AS temperature_c,
            ts,
            CAST(year  AS VARCHAR) AS year,
            CAST(month AS VARCHAR) AS month,
            CAST(day   AS VARCHAR) AS day,
            CAST(hour  AS VARCHAR) AS hour
        FROM read_json(
            's3://{MINIO_BUCKET}/clean/**/*.json',
            hive_partitioning = true,
            columns = {{
                device_id:     'VARCHAR',
                temperature_c: 'DOUBLE',
                ts:            'VARCHAR'
            }}
        )
        ORDER BY ts DESC
        LIMIT 10000
    """).df()
    print(f'Registros leídos de MinIO: {len(minio_df)}')
    print(minio_df.dtypes)
except Exception as e:
    print(f'No se pudo leer MinIO: {e}')
    minio_df = pd.DataFrame()
finally:
    conn.close()

In [ ]:
if minio_df.empty:
    print('Sin datos en MinIO. Ejecuta el pipeline y espera ~60 segundos.')
else:
    total = len(minio_df)

    # Análisis de nulos
    null_counts = minio_df.isnull().sum()
    null_pct    = (null_counts / total * 100).round(2)
    completitud = pd.DataFrame({'nulos': null_counts, 'pct_nulos': null_pct})
    completitud['pct_completo'] = (100 - null_pct).round(2)
    print('Completitud por campo:')
    print(completitud.to_string())

    # Registros completamente válidos
    valid_mask = minio_df['temperature_c'].notna() & minio_df['device_id'].notna()
    n_valid    = valid_mask.sum()
    print(f'\nRegistros válidos (temperature_c + device_id no nulos): {n_valid}/{total} ({n_valid/total*100:.1f}%)')

    # Gráfico de completitud
    fig_comp = px.bar(
        completitud.reset_index().rename(columns={'index': 'campo'}),
        x='campo',
        y='pct_completo',
        title='Completitud de campos en MinIO (cold path)',
        labels={'pct_completo': '% completo', 'campo': 'Campo'},
        color='pct_completo',
        color_continuous_scale='RdYlGn',
        range_color=[0, 100],
        template='plotly_dark',
    )
    fig_comp.update_layout(height=350)
    fig_comp.show()

---
## 4. Distribución de Unidades de Temperatura

El simulador genera mensajes en C, F y K según la configuración de cada máquina.
Aquí vemos la proporción real de unidades en `sensors_raw`
y comparamos con lo que llega normalizado a `sensors_clean`.

In [ ]:
if not raw_msgs:
    print('No hay mensajes en sensors_raw para analizar unidades.')
else:
    raw_df = pd.DataFrame(raw_msgs)
    if 'unit' not in raw_df.columns:
        print('Campo unit no encontrado en los mensajes.')
    else:
        unit_counts = raw_df['unit'].value_counts().reset_index()
        unit_counts.columns = ['unit', 'count']
        unit_counts['pct'] = (unit_counts['count'] / unit_counts['count'].sum() * 100).round(1)

        print('Distribución de unidades en sensors_raw:')
        print(unit_counts.to_string(index=False))

        fig_units = make_subplots(
            rows=1, cols=2,
            specs=[[{'type': 'pie'}, {'type': 'bar'}]],
            subplot_titles=['Proporción de unidades', 'Temperatura media por unidad (°C convertida)'],
        )

        fig_units.add_trace(
            go.Pie(
                labels=unit_counts['unit'],
                values=unit_counts['count'],
                hole=0.4,
                marker_colors=['#4CAF50', '#FF9800', '#2196F3'],
            ),
            row=1, col=1
        )

        # Temperatura media en °C por unidad original
        raw_df['temp_c'] = raw_df.apply(
            lambda r: to_celsius(r['temperature'], r['unit']) if pd.notna(r.get('temperature')) else None,
            axis=1
        )
        mean_by_unit = raw_df.groupby('unit')['temp_c'].mean().reset_index()
        mean_by_unit.columns = ['unit', 'mean_c']

        fig_units.add_trace(
            go.Bar(
                x=mean_by_unit['unit'],
                y=mean_by_unit['mean_c'].round(2),
                marker_color=['#4CAF50', '#FF9800', '#2196F3'],
                text=mean_by_unit['mean_c'].round(1),
                textposition='auto',
            ),
            row=1, col=2
        )

        fig_units.update_layout(
            title='Distribución de unidades de temperatura',
            height=400, template='plotly_dark', showlegend=False,
        )
        fig_units.show()

---
## 5. Validación de Mensajes con `validators.py`

Aplicamos `validate_raw_message()` al batch de mensajes leídos de `sensors_raw`
y comparamos con lo que Flink rechazó (DLQ).
Esto permite verificar que la lógica del bridge y la de Flink son consistentes.

In [ ]:
if not raw_msgs:
    print('No hay mensajes para validar.')
else:
    validation_results = []
    for msg in raw_msgs:
        ok, reason = validate_raw_message(msg)
        validation_results.append({
            'device_id': msg.get('device_id', '?'),
            'unit':      msg.get('unit', '?'),
            'ok':        ok,
            'reason':    reason if not ok else 'válido',
        })

    val_df = pd.DataFrame(validation_results)
    n_ok      = val_df['ok'].sum()
    n_invalid = (~val_df['ok']).sum()
    total_v   = len(val_df)

    print(f'Resultado validación local ({total_v} mensajes):')
    print(f'  Válidos  : {n_ok}  ({n_ok/total_v*100:.1f}%)')
    print(f'  Inválidos: {n_invalid}  ({n_invalid/total_v*100:.1f}%)')

    if n_invalid > 0:
        print('\nMotivos de rechazo (validación local):')
        print(val_df[~val_df['ok']]['reason'].value_counts().to_string())

    # Gráfico comparativo: validación local vs DLQ
    comparison = pd.DataFrame([
        {'fuente': 'Validación local (validators.py)', 'válidos': n_ok,         'rechazados': n_invalid},
        {'fuente': 'DLQ Flink (sensors_invalid)',      'válidos': total_v - len(dlq_msgs), 'rechazados': len(dlq_msgs)},
    ])

    fig_val = px.bar(
        comparison,
        x='fuente',
        y=['válidos', 'rechazados'],
        title='Comparativa: validación local vs Flink DLQ',
        labels={'value': 'Mensajes', 'variable': 'Estado'},
        color_discrete_map={'válidos': '#4caf50', 'rechazados': '#f44336'},
        barmode='group',
        template='plotly_dark',
    )
    fig_val.update_layout(height=380)
    fig_val.show()

---
## 6. Resumen Ejecutivo de Calidad de Datos

In [ ]:
print('=' * 60)
print('RESUMEN EJECUTIVO — CALIDAD DE DATOS')
print('=' * 60)

total_raw  = len(raw_msgs)
total_dlq  = len(dlq_msgs)
total_cold = len(minio_df) if not minio_df.empty else 0

# 1. Volumen
print(f'\n1. VOLUMEN')
print(f'   sensors_raw  (leídos): {total_raw}')
print(f'   sensors_invalid (DLQ): {total_dlq}')
print(f'   cold path (MinIO):     {total_cold}')

# 2. Integridad hash
if total_raw > 0 and 'integrity_df' in dir():
    ov_valid = integrity_df['valid'].sum()
    ov_total = integrity_df['total'].sum()
    ov_pct   = ov_valid / ov_total * 100 if ov_total > 0 else 0
    print(f'\n2. INTEGRIDAD HASH')
    print(f'   Mensajes verificados: {ov_valid}/{ov_total} ({ov_pct:.1f}%)')
    for _, row in integrity_df.iterrows():
        print(f'   {row["device_id"]}: {row["integrity_pct"]}% integridad')

# 3. Completitud MinIO
if not minio_df.empty:
    valid_cold = minio_df['temperature_c'].notna().sum()
    pct_cold   = valid_cold / total_cold * 100
    print(f'\n3. COMPLETITUD COLD PATH')
    print(f'   Registros con temperature_c: {valid_cold}/{total_cold} ({pct_cold:.1f}%)')

# 4. Tasa de rechazo
if total_raw > 0:
    tasa_rechazo = total_dlq / total_raw * 100
    estado = 'BUENA' if tasa_rechazo < 5 else ('ACEPTABLE' if tasa_rechazo < 15 else 'REVISAR')
    print(f'\n4. TASA DE RECHAZO DLQ')
    print(f'   {total_dlq}/{total_raw} ({tasa_rechazo:.1f}%)  → {estado}')

# 5. Gauge visual
metrics_for_gauge = []
if total_raw > 0:
    metrics_for_gauge.append(('Hash Integrity', ov_pct if 'ov_pct' in dir() else 100.0))
if not minio_df.empty:
    metrics_for_gauge.append(('Cold Completitud', valid_cold / total_cold * 100))
if total_raw > 0:
    metrics_for_gauge.append(('Aceptación DLQ', 100 - tasa_rechazo))

if metrics_for_gauge:
    n_gauges = len(metrics_for_gauge)
    specs    = [[{'type': 'indicator'}] * n_gauges]
    fig_gauge = make_subplots(
        rows=1, cols=n_gauges,
        specs=specs,
        subplot_titles=[m[0] for m in metrics_for_gauge],
    )
    for i, (label, value) in enumerate(metrics_for_gauge):
        color = 'green' if value >= 95 else ('orange' if value >= 80 else 'red')
        fig_gauge.add_trace(
            go.Indicator(
                mode='gauge+number',
                value=round(value, 1),
                number={'suffix': '%', 'font': {'size': 28}},
                gauge={
                    'axis': {'range': [0, 100]},
                    'bar':  {'color': color},
                    'steps': [
                        {'range': [0,   80], 'color': '#3d1a1a'},
                        {'range': [80,  95], 'color': '#3d3000'},
                        {'range': [95, 100], 'color': '#1a3d1a'},
                    ],
                    'threshold': {
                        'line': {'color': 'white', 'width': 2},
                        'thickness': 0.75,
                        'value': 95,
                    },
                },
            ),
            row=1, col=i + 1,
        )
    fig_gauge.update_layout(
        title='Indicadores globales de calidad',
        height=300, template='plotly_dark',
    )
    fig_gauge.show()

print('\n' + '=' * 60)